# Day Split — GréineGrid RL

This notebook prepares the merged dataset for RL training by:

1. Labelling each 30-minute row with its **episode day** (one RL episode = one calendar day, 48 steps)
2. Filtering out incomplete days and excluding customer 2 (shorter recording period)
3. Creating a reproducible **train / validation / test** split

**Input:** `data/merged_30min_v2.csv` (from `build_dataset.ipynb`)

**Output:** `data/day_split.json`

## 1. Load merged data and label episode days

Each RL episode spans one operating day. A one-minute offset aligns midnight-boundary intervals with the preceding trading day (same logic as `src/data_loader.py`).

In [ ]:
import pandas as pd

df = pd.read_csv("data/merged_30min_v2.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.tz_convert("Australia/Sydney")
df["episode_day"] = (df["timestamp"] - pd.Timedelta(minutes=1)).dt.date

steps = df.groupby(["customer_id", "episode_day"]).size()
incomplete_days = steps[steps != 48]

for customer in sorted(df["customer_id"].unique()):
    n = (incomplete_days.index.get_level_values("customer_id") == customer).sum()
    print(f"Customer {customer}: {n} days with steps != 48")

print()
print("Days per customer:")
print(df.groupby("customer_id")["episode_day"].nunique())

## 2. Filter to complete days

- **Customer 2** is excluded — only 284 days vs 365 for the others (different recording window).
- Days with fewer or more than 48 intervals are dropped so every episode has a fixed horizon.

In [ ]:
df = df[df["customer_id"] != 2].copy()

steps = df.groupby(["customer_id", "episode_day"])["timestamp"].transform("size")
df = df[steps == 48].copy()

print(f"Rows after filtering: {len(df):,}")
print(f"Unique episode days: {df['episode_day'].nunique()}")
print(f"Customers retained: {sorted(df['customer_id'].unique())}")

## 3. Stratified day split by month

Days are shuffled **within each calendar month** so train/val/test all see seasonal variation. Default ratios: 70% train, 15% validation, 15% test.

In [ ]:
import numpy as np


def split_days(days, seed=50, train_frac=0.70, val_frac=0.15):
    """Split calendar days into train/val/test, stratified by month."""
    rng = np.random.default_rng(seed=seed)
    frame = pd.DataFrame({"day": pd.Series(sorted(days))})
    frame["month"] = pd.to_datetime(frame["day"]).dt.strftime("%Y-%m")

    train, val, test = [], [], []
    for _, group in frame.groupby("month"):
        arr = group["day"].to_numpy()
        arr = arr[rng.permutation(len(arr))]
        n = len(arr)
        n_train = int(round(train_frac * n))
        n_val = int(round(val_frac * n))
        train += list(arr[:n_train])
        val += list(arr[n_train : n_train + n_val])
        test += list(arr[n_train + n_val :])
    return train, val, test

In [ ]:
train_days, val_days, test_days = split_days(df["episode_day"].unique())

total = df["episode_day"].nunique()
print(f"Train: {len(train_days)} | Val: {len(val_days)} | Test: {len(test_days)} | Total: {total}")
assert len(train_days) + len(val_days) + len(test_days) == total

## 4. Export split to JSON

In [ ]:
import json

split = {
    "seed": 50,
    "train": [str(d) for d in train_days],
    "val": [str(d) for d in val_days],
    "test": [str(d) for d in test_days],
}

with open("data/day_split.json", "w", encoding="utf-8") as f:
    json.dump(split, f, indent=2)

print(f"Saved data/day_split.json — {len(train_days)} train, {len(val_days)} val, {len(test_days)} test")

## 5. Quick sanity check

Confirm every split day exists in the filtered dataset with exactly 48 steps.

In [ ]:
for name, days in [("train", train_days), ("val", val_days), ("test", test_days)]:
    subset = df[df["episode_day"].isin(days)]
    n_days = subset["episode_day"].nunique()
    steps_ok = (subset.groupby("episode_day").size() == 48).all()
    print(f"{name:5s}: {n_days} days | all 48 steps: {steps_ok}")

---

**Next step:** train an agent with:

```bash
python -m src.train --agent q_learning
```

The training script reads `data/day_split.json` and samples random days from the train split.